# Step 4 — Welfare accounting and figure generation

This notebook combines the processed environmental cost files, computes net welfare metrics, and generates the main paper figures used in the manuscript.

## Inputs
- `../data/outputs/enteric_manure_emissions_cost_country_year.csv`
- `../data/outputs/land_opportunity_cost_country_year.csv`
- `../data/outputs/nitrogen_cost_livestock_country_year.csv`
- `../data/outputs/livestock_net_welfare_global_year.csv`
- `../data/outputs/livestock_net_welfare_country_year.csv`

## Outputs
- Main manuscript figures in `../figures/`
- Country and global summary tables in `../data/outputs/`

In [38]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

---------------------------------------------------------------
Paths
---------------------------------------------------------------

In [39]:
def get_paths():
    """Return outputs and figures directories.

    Works both as a .py script and after conversion to a Jupyter notebook.
    Assumes the file/notebook sits in repo/code/.
    """
    try:
        repo_root = Path(__file__).resolve().parents[1]
    except NameError:
        repo_root = Path.cwd().parent

    outputs = repo_root / "data" / "outputs"
    figures = repo_root / "figures"
    os.makedirs(figures, exist_ok=True)

    return outputs, figures

---------------------------------------------------------------
Style helpers
---------------------------------------------------------------

In [40]:
# ---------------------------------------------------------------
# Color scheme (Nature-style, color-blind safe)
# ---------------------------------------------------------------

# Economic aggregates
COL_PROD = "#b22222"      # production value (deep red)
COL_TOTAL = "#1f77b4"     # total external damages (blue)
COL_NET = "#000000"       # net welfare (black)

# Environmental drivers
COL_GHG = "#1f77b4"       # enteric & manure GHG
COL_LAND = "#ff8c00"      # land opportunity cost
COL_N = "#2ca02c"         # nitrogen pollution

# Scenario colors (used in ratio plots etc.)
SCEN_COLORS = {
    "low": "#4c78a8",
    "mid": "#f58518",
    "high": "#54a24b"
}

In [41]:
def trillions_formatter(x, pos=None):
    return f"{x/1e12:.1f}T"

In [42]:
def billions_formatter(x, pos=None):
    return f"{x/1e9:.0f}"

In [43]:
def save_figure(fig, figures, name):
    """Save both PNG and PDF versions at publication-friendly resolution."""
    png = figures / f"{name}.png"
    pdf = figures / f"{name}.pdf"
    fig.savefig(png, bbox_inches="tight", dpi=500)
    fig.savefig(pdf, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {pdf}")

In [44]:
def style_axes(ax):
    """Minimal axis styling similar to journal figures."""
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", labelsize=10)
    ax.grid(False)

In [45]:
def set_common_year_ticks(ax):
    """Use a clean 5-year interval for time-series figures."""
    ax.set_xticks([2000, 2005, 2010, 2015, 2020])

---------------------------------------------------------------
Data loading
---------------------------------------------------------------

In [65]:
def load_processed_data(outputs):
    """Load the processed welfare datasets."""
    global_df = pd.read_csv(outputs / "livestock_net_welfare_global_year.csv")
    country_df = pd.read_csv(outputs / "livestock_net_welfare_country_year.csv")

    # Ensure consistent ordering for panel figures.
    scen_order = ["low", "mid", "high"]
    global_df["scenario"] = pd.Categorical(global_df["scenario"], scen_order, ordered=True)
    country_df["scenario"] = pd.Categorical(country_df["scenario"], scen_order, ordered=True)

    global_df = global_df.sort_values(["scenario", "Year"]).reset_index(drop=True)
    country_df = country_df.sort_values(["scenario", "Year", "Area"]).reset_index(drop=True)

    return global_df, country_df

---------------------------------------------------------------
Figure 1. Global livestock welfare accounting
---------------------------------------------------------------

In [75]:
def make_figure_1(global_df, figures):
    """Three-panel time-series figure for low / mid / high scenarios."""
    scenarios = ["low", "mid", "high"]

    fig, axes = plt.subplots(
        1, 3, figsize=(14.5, 4.8), sharex=True, constrained_layout=True
    )

    panel_labels = ["a", "b", "c"]

    for ax, s, lab in zip(axes, scenarios, panel_labels):
        d = global_df[global_df["scenario"] == s].copy()

        ax.plot(d["Year"], d["livestock_value_usd"],
                color=COL_PROD, lw=2.8)

        ax.plot(d["Year"], d["total_external_cost_usd"],
                color=COL_TOTAL, lw=2.8)

        ax.plot(d["Year"], d["net_welfare_usd"],
                color=COL_NET, lw=2.8)

        ax.text(
            -0.08, 1.04, lab,
            transform=ax.transAxes,
            fontsize=15,
            fontweight="bold",
            va="bottom"
        )

        ax.set_title(f"Scenario: {s}", fontsize=14, loc="left", pad=10)

        set_common_year_ticks(ax)
        style_axes(ax)

        ax.tick_params(axis="both", labelsize=14)

        ax.yaxis.set_major_formatter(
            mtick.FuncFormatter(trillions_formatter)
        )

        ax.margins(x=0.02)

    axes[0].set_ylabel("USD (trillions)", fontsize=15)
    axes[1].set_xlabel("Year", fontsize=15)

    handles = [
        Line2D([0], [0], color=COL_PROD, lw=2.8,
               label="Production value"),
        Line2D([0], [0], color=COL_TOTAL, lw=2.8,
               label="Total external damages"),
        Line2D([0], [0], color=COL_NET, lw=2.8,
               label="Net welfare"),
    ]

    fig.legend(
        handles=handles,
        loc="upper center",
        ncol=3,
        frameon=False,
        fontsize=14,
        bbox_to_anchor=(0.5, 1.10)
    )

    save_figure(fig, figures,
                "Fig1_global_welfare_accounting_all_scenarios_NATURE")

---------------------------------------------------------------
Figure 2. Absolute decomposition of external damages
---------------------------------------------------------------

In [67]:
def make_figure_2(global_df, figures):
    """Mid-scenario decomposition into GHG, land, and nitrogen."""
    d = global_df[global_df["scenario"] == "mid"].copy()

    fig, ax = plt.subplots(figsize=(8.2, 4.4), constrained_layout=True)

    ax.stackplot(
        d["Year"],
        d["enteric_manure_cost_usd"],
        d["land_opportunity_cost_usd"],
        d["nitrogen_cost_usd"],
        labels=[
            "Enteric and manure GHG emissions (CO$_2$e)",
            "Land opportunity cost",
            "Nitrogen pollution",
        ],
        colors=[COL_GHG, COL_LAND, COL_N],
        alpha=0.95,
    )

    set_common_year_ticks(ax)
    style_axes(ax)
    ax.set_title("b", fontsize=11, loc="left", pad=6)
    ax.set_xlabel("Year", fontsize=11)
    ax.set_ylabel("USD (trillions)", fontsize=11)
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(trillions_formatter))
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, 1.15), ncol=3, frameon=False, fontsize=10)

    save_figure(fig, figures, "Fig2_damage_decomposition_mid_NATURE")

---------------------------------------------------------------
Figure 3. Net welfare with and without land opportunity cost
---------------------------------------------------------------

In [78]:
def make_figure_3(global_df, figures):
    """Three-panel comparison of full net welfare vs excluding land cost."""
    scenarios = ["low", "mid", "high"]

    fig, axes = plt.subplots(
        1, 3, figsize=(14.5, 4.8), sharex=True, constrained_layout=True
    )

    panel_labels = ["a", "b", "c"]

    for ax, s, lab in zip(axes, scenarios, panel_labels):
        d = global_df[global_df["scenario"] == s].copy()

        welfare_without_land = (
            d["livestock_value_usd"]
            - d["enteric_manure_cost_usd"]
            - d["nitrogen_cost_usd"]
        )

        ax.plot(d["Year"], d["net_welfare_usd"],
                color=COL_NET, lw=2.8)

        ax.plot(d["Year"], welfare_without_land,
                color=COL_PROD, lw=2.8, linestyle="--")

        ax.text(
            -0.08, 1.04, lab,
            transform=ax.transAxes,
            fontsize=15,
            fontweight="bold",
            va="bottom"
        )

        ax.set_title(f"Scenario: {s}", fontsize=14, loc="left", pad=10)

        set_common_year_ticks(ax)
        style_axes(ax)

        ax.tick_params(axis="both", labelsize=14)

        ax.yaxis.set_major_formatter(
            mtick.FuncFormatter(trillions_formatter)
        )

        ax.margins(x=0.02)

    axes[0].set_ylabel("USD (trillions)", fontsize=15)
    axes[1].set_xlabel("Year", fontsize=15)

    handles = [
        Line2D([0], [0], color=COL_NET, lw=2.8,
               label="Net welfare (full damages)"),
        Line2D([0], [0], color=COL_PROD, lw=2.8, linestyle="--",
               label="Net welfare (excluding land opportunity)"),
    ]

    fig.legend(
        handles=handles,
        loc="upper center",
        ncol=2,
        frameon=False,
        fontsize=14,
        bbox_to_anchor=(0.5, 1.10)
    )

    save_figure(fig, figures,
                "Fig3_with_vs_without_land_all_scenarios_NATURE")

---------------------------------------------------------------
Figure 4. External damages / production value ratio
---------------------------------------------------------------

In [69]:
def make_figure_4(global_df, figures):
    """One-panel scenario comparison of damage-to-benefit ratio."""
    fig, ax = plt.subplots(figsize=(7.0, 4.5), constrained_layout=True)

    for s in ["low", "mid", "high"]:
        d = global_df[global_df["scenario"] == s].copy()
        ax.plot(d["Year"], d["cost_to_benefit_ratio"], lw=2.3, color=SCEN_COLORS[s], label=s)

    set_common_year_ticks(ax)
    style_axes(ax)
    ax.set_xlabel("Year", fontsize=11)
    ax.set_ylabel("External damages / production value", fontsize=11)
    ax.legend(title="Scenario", frameon=False, loc="upper center", bbox_to_anchor=(0.5, 1.12), ncol=3, fontsize=10, title_fontsize=10)

    save_figure(fig, figures, "Fig4_cost_benefit_ratio_all_scenarios_NATURE")

---------------------------------------------------------------
Country figure helper
---------------------------------------------------------------

In [70]:
def _prepare_country_snapshot(country_df, scenario="mid", year=2022, n=20):
    """Filter to one scenario-year and return top countries by total damages."""
    snap = country_df[(country_df["scenario"] == scenario) & (country_df["Year"] == year)].copy()
    snap = snap.sort_values("total_external_cost_usd", ascending=False).head(n).copy()

    # Reverse so the largest country appears at the top in a horizontal plot.
    snap = snap.iloc[::-1].reset_index(drop=True)
    return snap

---------------------------------------------------------------
Figure 5. Country stacked losses with production marker
---------------------------------------------------------------

In [71]:
def make_country_top_losses(country_df, figures, scenario="mid", year=2022, n=20):
    """Horizontal stacked bars with production value shown as a thin marker."""
    snap = _prepare_country_snapshot(country_df, scenario=scenario, year=year, n=n)

    y = np.arange(len(snap))

    fig_h = max(6.8, 0.34 * len(snap) + 1.6)
    fig, ax = plt.subplots(figsize=(10.5, fig_h), constrained_layout=True)

    # Stacked damage bars
    ghg = snap["enteric_manure_cost_usd"].to_numpy()
    land = snap["land_opportunity_cost_usd"].to_numpy()
    nit = snap["nitrogen_cost_usd"].to_numpy()
    prod = snap["livestock_value_usd"].to_numpy()

    ax.barh(y, ghg, color=COL_GHG, edgecolor="none", label="Enteric & manure GHG (CO$_2$e)")
    ax.barh(y, land, left=ghg, color=COL_LAND, edgecolor="none", label="Land opportunity cost")
    ax.barh(y, nit, left=ghg + land, color=COL_N, edgecolor="none", label="Nitrogen pollution")

    # Production value marker as a thin vertical line centered on each bar
    marker_half_height = 0.42
    for yi, pv in zip(y, prod):
        ax.vlines(pv, yi - marker_half_height, yi + marker_half_height, color=COL_PROD, linewidth=2.4, zorder=5)

    ax.set_yticks(y)
    ax.set_yticklabels(snap["Area"], fontsize=10)
    ax.set_xlabel("USD (billion, 2014–2016 constant where applicable)", fontsize=11)
    ax.xaxis.set_major_formatter(mtick.FuncFormatter(billions_formatter))
    style_axes(ax)

    # Give a bit of room on the right so long bars/markers do not hit the frame.
    xmax = np.max(ghg + land + nit)
    ax.set_xlim(0, xmax * 1.06)

    # Legend matching the publication figure.
    handles = [
        Line2D([0], [0], color=COL_PROD, marker="|", markersize=16, linestyle="None", markeredgewidth=2.5, label="Production value"),
        Patch(facecolor=COL_GHG, label="Enteric & manure GHG (CO$_2$e)"),
        Patch(facecolor=COL_LAND, label="Land opportunity cost"),
        Patch(facecolor=COL_N, label="Nitrogen pollution"),
    ]
    ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 1.10), ncol=2, frameon=False, fontsize=10)

    save_figure(fig, figures, f"Fig_country_top_losses_stacked_drivers_{year}_{scenario}_NATURE")

---------------------------------------------------------------
Extended Data. Country welfare loss intensity
---------------------------------------------------------------

In [72]:
def make_country_loss_intensity(country_df, figures, scenario="mid", year=2022, n=20):
    """Horizontal ranking of welfare loss per $ of output."""
    snap = country_df[(country_df["scenario"] == scenario) & (country_df["Year"] == year)].copy()

    # Define intensity as positive loss per $ of output.
    snap["loss_intensity"] = (-snap["net_welfare_usd"] / snap["livestock_value_usd"]).clip(lower=0)

    snap = snap.sort_values("loss_intensity", ascending=False).head(n).copy()
    snap = snap.iloc[::-1].reset_index(drop=True)

    y = np.arange(len(snap))
    fig_h = max(6.8, 0.34 * len(snap) + 1.6)
    fig, ax = plt.subplots(figsize=(8.8, fig_h), constrained_layout=True)

    ax.barh(y, snap["loss_intensity"], color=COL_LAND, edgecolor="none")

    ax.set_yticks(y)
    ax.set_yticklabels(snap["Area"], fontsize=10)
    ax.set_xlabel("Welfare loss per $ of output (-net welfare / production value)", fontsize=11)
    style_axes(ax)

    xmax = snap["loss_intensity"].max()
    ax.set_xlim(0, xmax * 1.08)

    save_figure(fig, figures, f"ExtData_country_loss_intensity_top20_{year}_{scenario}_NATURE")

---------------------------------------------------------------
Main
---------------------------------------------------------------

In [79]:
def main():
    outputs, figures = get_paths()
    global_df, country_df = load_processed_data(outputs)

    print("Generating Nature-style figures...")
    make_figure_1(global_df, figures)
    make_figure_2(global_df, figures)
    make_figure_3(global_df, figures)
    make_figure_4(global_df, figures)
    make_country_top_losses(country_df, figures, scenario="mid", year=2022, n=20)
    make_country_loss_intensity(country_df, figures, scenario="mid", year=2022, n=20)
    print("Done.")

In [80]:
if __name__ == "__main__":
    main()

Generating Nature-style figures...
Saved: /Users/ritukajaiswal/Desktop/Economic_solution_AF/Economic_analysis_25_feb/figures/Fig1_global_welfare_accounting_all_scenarios_NATURE.pdf
Saved: /Users/ritukajaiswal/Desktop/Economic_solution_AF/Economic_analysis_25_feb/figures/Fig2_damage_decomposition_mid_NATURE.pdf
Saved: /Users/ritukajaiswal/Desktop/Economic_solution_AF/Economic_analysis_25_feb/figures/Fig3_with_vs_without_land_all_scenarios_NATURE.pdf
Saved: /Users/ritukajaiswal/Desktop/Economic_solution_AF/Economic_analysis_25_feb/figures/Fig4_cost_benefit_ratio_all_scenarios_NATURE.pdf
Saved: /Users/ritukajaiswal/Desktop/Economic_solution_AF/Economic_analysis_25_feb/figures/Fig_country_top_losses_stacked_drivers_2022_mid_NATURE.pdf
Saved: /Users/ritukajaiswal/Desktop/Economic_solution_AF/Economic_analysis_25_feb/figures/ExtData_country_loss_intensity_top20_2022_mid_NATURE.pdf
Done.
